# Layout model evaluation — error analysis

Analysis of the hand-made error log `evaluation/boxing_error_log_template.csv`
(see `documentation/evaluation_error_log_documentation.md` for how it was filled in).

**Counting rules used throughout**

- `good` and `good abandon` are correct boxes; everything else is an error verdict.
- A combined verdict (e.g. `split; missed region`) is split into its parts, so it counts as
  **one error per part** (2 errors here) but still **one incorrect box**.
- `misplacement abandon` is counted as a `misplacement` error (it is flagged in `is_abandon`).
- A box with a second row (error spotted later) contributes the errors from both rows, but is one box.
- **Errors** = error instances. **Boxes** = unique `(magazine, page, model, block_index)`.
- YOLO duplicate boxes: the second copy has `double of N` in its notes. Every function takes
  `drop_duplicates=True` to report without those copies.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 140)

# Notebook lives in evaluation/, but work from either the repo root or evaluation/.
LOG_PATH = next(p for p in [Path("boxing_error_log_template.csv"), Path("evaluation/boxing_error_log_template.csv")] if p.exists())

MODELS = ["google", "prima", "yolo"]
MAGAZINES = ["chambers", "penny", "saturday"]
MODEL_NAMES = {"google": "Document AI", "prima": "PrimaLayout", "yolo": "DocLayout-YOLO"}
CORRECT = {"good", "good abandon"}
ERROR_TYPES = ["split", "merge", "under-coverage", "misplacement", "missed region", "mislabel"]

## Load and tidy the log

In [2]:
def load_log(path=LOG_PATH):
    """One row per CSV row, with helper columns added."""
    df = pd.read_csv(path, dtype={"notes": str}).fillna({"notes": ""})
    df["error_type"] = df["error_type"].str.strip()
    df["is_correct"] = df["error_type"].isin(CORRECT)
    df["is_abandon"] = df["error_type"].str.contains("abandon")
    df["is_duplicate"] = df["notes"].str.contains("double of", case=False)
    df["box_id"] = df[["magazine", "page", "model", "block_index"]].astype(str).agg("|".join, axis=1)
    return df


def explode_errors(df):
    """One row per error instance (correct verdicts dropped, combos split apart)."""
    errs = df[~df["is_correct"]].copy()
    errs["error"] = errs["error_type"].str.replace(" abandon", "", regex=False).str.split(";")
    errs = errs.explode("error")
    errs["error"] = errs["error"].str.strip()
    return errs


def _filter(df, model=None, drop_duplicates=False):
    if drop_duplicates:
        df = df[~df["is_duplicate"]]
    if model is not None:
        df = df[df["model"] == model]
    return df


log = load_log()
unknown = set(explode_errors(log)["error"]) - set(ERROR_TYPES)
assert not unknown, f"Unrecognised error types in log: {unknown}"
print(f"{len(log)} rows, {log['box_id'].nunique()} boxes, {log['is_duplicate'].sum()} YOLO duplicate copies")
log.head()

495 rows, 482 boxes, 11 YOLO duplicate copies


,magazine,page,model,block_index,error_type,notes,is_correct,is_abandon,is_duplicate,box_id
0,chambers,1,google,0,good abandon,,True,True,False,chambers|1|google|0
1,chambers,1,google,1,good abandon,,True,True,False,chambers|1|google|1
2,chambers,1,google,2,split,masthead image,False,False,False,chambers|1|google|2
3,chambers,1,google,3,split; merge,masthead image and author text,False,False,False,chambers|1|google|3
4,chambers,1,google,4,good,,True,False,False,chambers|1|google|4


## 1. Total errors per model

In [3]:
def total_errors(model=None, drop_duplicates=False, df=None):
    """Errors, incorrect boxes, total boxes and rates per model (or for one model)."""
    df = _filter(log if df is None else df, model, drop_duplicates)
    errs = explode_errors(df)
    boxes = df.groupby("model")["box_id"].nunique()
    bad_boxes = df[~df["is_correct"]].groupby("model")["box_id"].nunique()
    good_boxes = boxes - bad_boxes.reindex(boxes.index, fill_value=0)
    pages = df.groupby("model")[["magazine", "page"]].apply(lambda g: len(g.drop_duplicates()))
    out = pd.DataFrame({
        "boxes": boxes,
        "errors": errs.groupby("model").size(),
        "incorrect_boxes": bad_boxes,
        "correct_boxes": good_boxes,
        "errors_per_page": errs.groupby("model").size() / pages,
        "pct_boxes_correct": 100 * good_boxes / boxes,
    }).fillna(0)
    out = out.reindex([m for m in MODELS if m in out.index]).rename(index=MODEL_NAMES)
    return out.round(1).astype({"boxes": int, "errors": int, "incorrect_boxes": int, "correct_boxes": int})


total_errors()

,boxes,errors,incorrect_boxes,correct_boxes,errors_per_page,pct_boxes_correct
model,,,,,,
Document AI,151,71,67,84,7.9,55.6
PrimaLayout,140,102,82,58,11.3,41.4
DocLayout-YOLO,191,89,87,104,9.9,54.5


In [4]:
# Without the second copy of YOLO's duplicate boxes
total_errors(drop_duplicates=True)

,boxes,errors,incorrect_boxes,correct_boxes,errors_per_page,pct_boxes_correct
model,,,,,,
Document AI,151,71,67,84,7.9,55.6
PrimaLayout,140,102,82,58,11.3,41.4
DocLayout-YOLO,180,85,83,97,9.4,53.9


In [5]:
# A single model
total_errors("yolo")

,boxes,errors,incorrect_boxes,correct_boxes,errors_per_page,pct_boxes_correct
model,,,,,,
DocLayout-YOLO,191,89,87,104,9.9,54.5


## 2. Errors by type per model

In [6]:
def errors_by_type(model=None, drop_duplicates=False, normalize=False, df=None):
    """Error-type x model table. normalize=True gives % of that model's errors."""
    errs = explode_errors(_filter(log if df is None else df, model, drop_duplicates))
    table = pd.crosstab(errs["error"], errs["model"])
    table = table.reindex(index=[e for e in ERROR_TYPES if e in table.index],
                          columns=[m for m in MODELS if m in table.columns], fill_value=0)
    if normalize:
        table = (100 * table / table.sum()).round(1)
    else:
        table.loc["TOTAL"] = table.sum()
    return table.rename(columns=MODEL_NAMES)


errors_by_type()

model,Document AI,PrimaLayout,DocLayout-YOLO
error,,,
split,43,74,82
merge,7,1,0
under-coverage,2,9,1
misplacement,15,3,2
missed region,4,15,1
mislabel,0,0,3
TOTAL,71,102,89


In [7]:
errors_by_type(normalize=True)

model,Document AI,PrimaLayout,DocLayout-YOLO
error,,,
split,60.6,72.5,92.1
merge,9.9,1.0,0.0
under-coverage,2.8,8.8,1.1
misplacement,21.1,2.9,2.2
missed region,5.6,14.7,1.1
mislabel,0.0,0.0,3.4


In [8]:
errors_by_type("prima")

model,PrimaLayout
error,
split,74
merge,1
under-coverage,9
misplacement,3
missed region,15
TOTAL,102


## 3. Errors per model by magazine — which magazine is hardest to box?

Raw error counts favour models that draw few boxes, so the error **rate** (errors per box) and
**% of boxes correct** are shown alongside.

In [9]:
def errors_by_magazine(model=None, drop_duplicates=False, metric="errors", df=None):
    """Magazine x model table.

    metric: "errors" (count), "errors_per_box", "pct_boxes_correct" or "boxes".
    """
    df = _filter(log if df is None else df, model, drop_duplicates)
    errs = explode_errors(df)
    keys = ["magazine", "model"]
    boxes = df.groupby(keys)["box_id"].nunique()
    n_err = errs.groupby(keys).size().reindex(boxes.index, fill_value=0)
    bad = df[~df["is_correct"]].groupby(keys)["box_id"].nunique().reindex(boxes.index, fill_value=0)
    values = {
        "errors": n_err,
        "boxes": boxes,
        "errors_per_box": (n_err / boxes).round(2),
        "pct_boxes_correct": (100 * (boxes - bad) / boxes).round(1),
    }[metric]
    table = values.unstack("model")
    table = table.reindex(index=[m for m in MAGAZINES if m in table.index],
                          columns=[m for m in MODELS if m in table.columns])
    if metric == "errors":
        table["ALL MODELS"] = table.sum(axis=1)
    return table.rename(columns=MODEL_NAMES)


def hardest_magazine(drop_duplicates=False):
    """Rank magazines by error rate pooled across all models."""
    df = _filter(log, drop_duplicates=drop_duplicates)
    errs = explode_errors(df)
    rank = pd.DataFrame({
        "boxes": df.groupby("magazine")["box_id"].nunique(),
        "errors": errs.groupby("magazine").size(),
    })
    rank["errors_per_box"] = (rank["errors"] / rank["boxes"]).round(2)
    rank["errors_per_page"] = (rank["errors"] / (3 * len(MODELS))).round(1)
    return rank.sort_values("errors_per_box", ascending=False)


errors_by_magazine()

model,Document AI,PrimaLayout,DocLayout-YOLO,ALL MODELS
magazine,,,,
chambers,32,45,37,114
penny,12,32,30,74
saturday,27,25,22,74


In [10]:
errors_by_magazine(metric="errors_per_box")

model,Document AI,PrimaLayout,DocLayout-YOLO
magazine,,,
chambers,0.56,1.02,0.55
penny,0.29,0.73,0.47
saturday,0.52,0.48,0.37


In [11]:
errors_by_magazine(metric="pct_boxes_correct")

model,Document AI,PrimaLayout,DocLayout-YOLO
magazine,,,
chambers,45.6,27.3,47.8
penny,73.8,40.9,53.1
saturday,51.9,53.8,63.3


In [12]:
hardest_magazine()

,boxes,errors,errors_per_box,errors_per_page
magazine,,,,
chambers,168,114,0.68,12.7
penny,150,74,0.49,8.2
saturday,164,74,0.45,8.2


In [13]:
# Which error types drive each magazine's difficulty?
errs = explode_errors(log)
pd.crosstab([errs["magazine"], errs["model"]], errs["error"]).reindex(columns=ERROR_TYPES, fill_value=0)

error            split  merge  under-coverage  misplacement  missed region  mislabel
magazine model                                                                      
chambers google     21      1               0             7              3         0
         prima      28      0               5             1             11         0
         yolo       33      0               0             2              0         2
penny    google      8      1               1             1              1         0
         prima      23      1               3             2              3         0
         yolo       29      0               1             0              0         0
saturday google     14      5               1             7              0         0
         prima      23      0               1             0              1         0
         yolo       20      0               0             0              1         1

## Extra views

### Per page
Where errors cluster on specific pages (e.g. one dense page driving a magazine's total).

In [14]:
def errors_by_page(metric="errors", drop_duplicates=False):
    df = _filter(log, drop_duplicates=drop_duplicates)
    errs = explode_errors(df)
    keys = ["magazine", "page", "model"]
    boxes = df.groupby(keys)["box_id"].nunique()
    n_err = errs.groupby(keys).size().reindex(boxes.index, fill_value=0)
    values = n_err if metric == "errors" else (n_err / boxes).round(2)
    return values.unstack("model")[MODELS].rename(columns=MODEL_NAMES)


errors_by_page()

model          Document AI  PrimaLayout  DocLayout-YOLO
magazine page                                          
chambers 1               7           10               7
         2               9           15              11
         3              16           20              19
penny    1               1            9               8
         2               7           11              10
         3               4           12              12
saturday 1              10            7               8
         2              10           10               9
         3               7            8               5

### What is causing the errors? (keywords in the notes)
The notes column records the cause and region type. This tags each error by keyword so causes can be
counted — edit `NOTE_TAGS` to add more.

In [15]:
NOTE_TAGS = {
    "indent": r"indent",
    "column": r"column",
    "masthead": r"masthead",
    "title": r"title",
    "page number": r"page number",
    "quote": r"quote",
    "poem": r"poem",
    "image": r"image",
    "caused by other misplaced box": r"misplac",
}


def errors_by_note_tag(error=None, drop_duplicates=False):
    """Count error instances whose notes match each tag (an error can match several tags)."""
    errs = explode_errors(_filter(log, drop_duplicates=drop_duplicates))
    if error is not None:
        errs = errs[errs["error"] == error]
    rows = {tag: errs[errs["notes"].str.contains(pat, case=False)].groupby("model").size()
            for tag, pat in NOTE_TAGS.items()}
    table = pd.DataFrame(rows).T.reindex(columns=MODELS).fillna(0).astype(int)
    table["ALL"] = table.sum(axis=1)
    return table.sort_values("ALL", ascending=False).rename(columns=MODEL_NAMES)


errors_by_note_tag("split")

model,Document AI,PrimaLayout,DocLayout-YOLO,ALL
indent,16,60,76,152
column,13,46,65,124
quote,3,5,12,20
caused by other misplaced box,12,0,0,12
title,5,0,2,7
image,4,1,2,7
masthead,4,0,0,4
poem,0,3,0,3
page number,0,0,0,0


In [16]:
errors_by_note_tag()

model,Document AI,PrimaLayout,DocLayout-YOLO,ALL
indent,17,69,77,163
column,25,56,65,146
title,11,6,6,23
quote,4,6,13,23
caused by other misplaced box,16,0,0,16
image,8,3,4,15
masthead,6,2,3,11
poem,2,3,0,5
page number,0,0,0,0


### Content vs page furniture
For extracting article text, a correct box around a page number matters less than a correct box around
an article. This separates furniture (`abandon` verdicts) from content boxes.

In [17]:
def content_vs_furniture(drop_duplicates=False):
    df = _filter(log, drop_duplicates=drop_duplicates).copy()
    df["kind"] = df["is_abandon"].map({True: "furniture", False: "content"})
    per_box = df.groupby(["model", "kind", "box_id"])["is_correct"].all().reset_index()
    table = per_box.groupby(["model", "kind"])["is_correct"].agg(boxes="size", correct="sum")
    table["pct_correct"] = (100 * table["correct"] / table["boxes"]).round(1)
    return table.unstack("kind").reindex(MODELS).rename(index=MODEL_NAMES)


content_vs_furniture()

boxes           correct           pct_correct          
kind           content furniture content furniture     content furniture
model                                                                   
Document AI         97        54      30        54        30.9     100.0
PrimaLayout        101        39      19        39        18.8     100.0
DocLayout-YOLO     132        59      47        57        35.6      96.6

## Fragmentation against the gold standard

Boxes per page compared with the hand-drawn gold standard in `evaluation/groundtruth_texts/`.
The gold boxes exist only as drawings on the PNGs, so the counts below were read off those images.
`furniture` = gold boxes labelled header/footer abandon; every other gold box (masthead, title,
byline, text) is `content`, which matches how `abandon` verdicts are split in the error log.

**Fragmentation ratio** = model boxes / gold boxes. 1.0 means the same number of boxes as the gold
standard; above 1 means over-segmentation (splitting); below 1 means merging or missed regions. It
is a count only: a ratio of 1.0 does not mean the boxes are the right ones. Use it with the error log.

In [18]:
# (magazine, page): (content boxes, furniture boxes), counted from the gold-standard PNGs
GOLD_BOXES = {
    ("chambers", 1): (8, 0),
    ("chambers", 2): (6, 1),
    ("chambers", 3): (5, 1),
    ("penny", 1): (9, 1),
    ("penny", 2): (4, 1),
    ("penny", 3): (8, 2),
    ("saturday", 1): (6, 1),
    ("saturday", 2): (4, 1),
    ("saturday", 3): (13, 2),
}
gold = pd.DataFrame(
    [(m, p, c, f) for (m, p), (c, f) in GOLD_BOXES.items()],
    columns=["magazine", "page", "gold_content", "gold_furniture"],
)
gold["gold_total"] = gold["gold_content"] + gold["gold_furniture"]


def boxes_per_page(drop_duplicates=False):
    """Model box counts per page (total and content-only) next to the gold standard."""
    df = _filter(log, drop_duplicates=drop_duplicates)
    # A box counts as furniture if any of its rows has an abandon verdict.
    per_box = df.groupby(["magazine", "page", "model", "box_id"])["is_abandon"].any().reset_index()
    counts = per_box.groupby(["magazine", "page", "model"]).agg(
        total=("box_id", "size"), content=("is_abandon", lambda a: (~a).sum()))
    counts = counts.unstack("model").reindex(columns=MODELS, level=1)
    counts.columns = [f"{model}_{kind}" for kind, model in counts.columns]
    return gold.merge(counts.reset_index(), on=["magazine", "page"])


def fragmentation(kind="content", by="page", drop_duplicates=False):
    """Model boxes / gold boxes.

    kind: "content" (excludes header/footer furniture) or "total".
    by: "page", "magazine" or "model" (pooled over all 9 pages).
    """
    bpp = boxes_per_page(drop_duplicates)
    gold_col = f"gold_{kind}"
    cols = [gold_col] + [f"{m}_{kind}" for m in MODELS]
    if by == "page":
        table = bpp.set_index(["magazine", "page"])[cols]
    elif by == "magazine":
        table = bpp.groupby("magazine")[cols].sum()
    else:
        table = bpp[cols].sum().to_frame("ALL PAGES").T
    ratio = table[[f"{m}_{kind}" for m in MODELS]].div(table[gold_col], axis=0).round(2)
    ratio.columns = [MODEL_NAMES[m] for m in MODELS]
    table.columns = ["gold"] + [f"{MODEL_NAMES[m]} boxes" for m in MODELS]
    return pd.concat([table, ratio.add_suffix(" ratio")], axis=1)


boxes_per_page()

,magazine,page,gold_content,gold_furniture,gold_total,google_total,prima_total,yolo_total,google_content,prima_content,yolo_content
0,chambers,1,8,0,8,17,12,20,11,9,16
1,chambers,2,6,1,7,17,14,22,12,11,15
2,chambers,3,5,1,6,23,18,25,17,14,19
3,penny,1,9,1,10,14,13,21,10,9,16
4,penny,2,4,1,5,12,13,18,8,9,11
5,penny,3,8,2,10,16,18,25,8,13,17
6,saturday,1,6,1,7,20,13,19,15,9,12
7,saturday,2,4,1,5,16,16,17,9,10,10
8,saturday,3,13,2,15,16,23,24,7,17,16


In [19]:
# Content boxes only, page by page
fragmentation()

gold  Document AI boxes  PrimaLayout boxes  DocLayout-YOLO boxes  Document AI ratio  PrimaLayout ratio  \
magazine page                                                                                                           
chambers 1        8                 11                  9                    16               1.38               1.12   
         2        6                 12                 11                    15               2.00               1.83   
         3        5                 17                 14                    19               3.40               2.80   
penny    1        9                 10                  9                    16               1.11               1.00   
         2        4                  8                  9                    11               2.00               2.25   
         3        8                  8                 13                    17               1.00               1.62   
saturday 1        6                 15                  9                    12               2.50               1.50   
         2        4                  9                 10                    10               2.25               2.50   
         3       13                  7                 17                    16               0.54               1.31   

               DocLayout-YOLO ratio  
magazine page                        
chambers 1                     2.00  
         2                     2.50  
         3                     3.80  
penny    1                     1.78  
         2                     2.75  
         3                     2.12  
saturday 1                     2.00  
         2                     2.50  
         3                     1.23

In [20]:
fragmentation(by="magazine")

,gold,Document AI boxes,PrimaLayout boxes,DocLayout-YOLO boxes,Document AI ratio,PrimaLayout ratio,DocLayout-YOLO ratio
magazine,,,,,,,
chambers,19,40,34,50,2.11,1.79,2.63
penny,21,26,31,44,1.24,1.48,2.10
saturday,23,31,36,38,1.35,1.57,1.65


In [21]:
fragmentation(by="model")

,gold,Document AI boxes,PrimaLayout boxes,DocLayout-YOLO boxes,Document AI ratio,PrimaLayout ratio,DocLayout-YOLO ratio
ALL PAGES,63,97,101,132,1.54,1.6,2.1


In [22]:
# All boxes, including headers / footers / page numbers
fragmentation(kind="total", by="model")

,gold,Document AI boxes,PrimaLayout boxes,DocLayout-YOLO boxes,Document AI ratio,PrimaLayout ratio,DocLayout-YOLO ratio
ALL PAGES,73,151,140,191,2.07,1.92,2.62


In [23]:
# YOLO without its duplicate second copies
fragmentation(by="model", drop_duplicates=True)

,gold,Document AI boxes,PrimaLayout boxes,DocLayout-YOLO boxes,Document AI ratio,PrimaLayout ratio,DocLayout-YOLO ratio
ALL PAGES,63,97,101,123,1.54,1.6,1.95
